The Layers of the Embedding Module will be:

Layer 1: Input Interface

Layer 2: Validation

Layer 3: Preprocessing

Layer 4: Encoding Engine

Layer 5: Output Vector

In [6]:
import pandas as pd
import os
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

FILE_PATH = r"D:\chiselon\Week 0\Week_0_Prep_Week_Ssample Data_clinic_cases.csv"

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError("CSV file not found at given path.")

df = pd.read_csv(FILE_PATH)

print("Dataset Loaded Successfully!")
print("Total Cases:", len(df))
print("Columns:", df.columns.tolist())


def preprocess_text(text: str) -> str:
    """
    Clean and normalize input text.
    """
    if not isinstance(text, str):
        raise TypeError("Input must be a string.")

    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)

    if text == "":
        raise ValueError("Input text cannot be empty after cleaning.")

    return text

def combine_text(symptoms: str, doctor_notes: str) -> str:
    """
    Combine symptoms and doctor notes into single text.
    """
    symptoms_clean = preprocess_text(symptoms)
    notes_clean = preprocess_text(doctor_notes)
    return symptoms_clean + " " + notes_clean

class EmbeddingEngine:
    """
    Modular TF-IDF based embedding generator.
    No similarity, no ML classification.
    Only encoding functionality.
    """

    def __init__(self, dataframe):
        if dataframe.empty:
            raise ValueError("Input dataframe is empty.")

        if 'symptoms' not in dataframe.columns or 'doctor_notes' not in dataframe.columns:
            raise ValueError("Required columns missing: 'symptoms', 'doctor_notes'")

        # Build corpus
        corpus = []

        for _, row in dataframe.iterrows():
            combined = combine_text(row['symptoms'], row['doctor_notes'])
            corpus.append(combined)

        # Initialize vectorizer once
        self.vectorizer = TfidfVectorizer()
        self.vectorizer.fit(corpus)

    def generate_embedding(self, text: str) -> np.ndarray:
        """
        Generate embedding vector for any input text.
        """

        # Validation
        if text is None:
            raise ValueError("Input text cannot be None.")

        if not isinstance(text, str):
            raise TypeError("Input must be a string.")

        cleaned_text = preprocess_text(text)

        # Transform
        vector = self.vectorizer.transform([cleaned_text])

        return vector.toarray()[0]

if __name__ == "__main__":

    print("\nInitializing Embedding Engine...\n")

    engine = EmbeddingEngine(df)

    # 3 Sample Cases
    sample_cases = [
        combine_text(
            "Severe chest pain radiating to arm",
            "Patient sweating and breathless. Known hypertension."
        ),
        combine_text(
            "High fever and persistent cough",
            "Patient reports fatigue and mild chest congestion."
        ),
        combine_text(
            "Severe headache and vomiting",
            "Patient sensitive to light. No prior migraine history."
        )
    ]

    print("Generating Embeddings for 3 Sample Cases:\n")

    for i, case in enumerate(sample_cases, 1):
        embedding = engine.generate_embedding(case)

        print(f"Case {i}:")
        print(f"Embedding Shape: {embedding.shape}")
        print(f"First 5 Values: {embedding[:5]}")
        print("-" * 50)

Dataset Loaded Successfully!
Total Cases: 10
Columns: ['case_id', 'clinic_id', 'symptoms', 'duration_days', 'doctor_notes', 'diagnosis', 'treatment', 'outcome', 'recovery_days', 'patient.age', 'patient.gender']

Initializing Embedding Engine...

Generating Embeddings for 3 Sample Cases:

Case 1:
Embedding Shape: (47,)
First 5 Values: [0.         0.         0.         0.59677497 0.        ]
--------------------------------------------------
Case 2:
Embedding Shape: (47,)
First 5 Values: [0. 0. 0. 1. 0.]
--------------------------------------------------
Case 3:
Embedding Shape: (47,)
First 5 Values: [0.         0.         0.         0.59677497 0.        ]
--------------------------------------------------


In [8]:
import os
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer


# TEXT PREPROCESSING


def preprocess_text(text: str) -> str:
    """
    Clean and normalize input text.
    Ensures consistent preprocessing across the system.
    """

    if text is None:
        raise ValueError("Input text cannot be None.")

    if not isinstance(text, str):
        raise TypeError("Input must be of type string.")

    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)              
    text = re.sub(r"[^a-z0-9\s]", "", text)       

    if text == "":
        raise ValueError("Input text is empty after preprocessing.")

    return text

# EMBEDDING ENGINE CLASS

class EmbeddingEngine:
    """
    Reusable embedding engine.
    Converts text → numerical vector using TF-IDF.
    """

    def __init__(self, corpus: list[str]):
        """
        Initialize the vectorizer using a corpus.
        """

        if not isinstance(corpus, list) or len(corpus) == 0:
            raise ValueError("Corpus must be a non-empty list of strings.")

        # Clean corpus
        cleaned_corpus = [preprocess_text(text) for text in corpus]

        self.vectorizer = TfidfVectorizer()
        self.vectorizer.fit(cleaned_corpus)

    def generate_embedding(self, text: str) -> np.ndarray:
        """
        Convert input text into embedding vector.
        """

        cleaned_text = preprocess_text(text)

        vector = self.vectorizer.transform([cleaned_text])

        return vector.toarray()[0]

# DATA LOADER (No Hardcoded Logic Inside Functions)

def load_corpus_from_csv(file_path: str,
                         symptoms_col: str,
                         notes_col: str) -> list[str]:
    """
    Load and combine symptoms + doctor notes into corpus list.
    """

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    df = pd.read_csv(file_path)

    if symptoms_col not in df.columns:
        raise ValueError(f"Column '{symptoms_col}' not found in CSV.")

    if notes_col not in df.columns:
        raise ValueError(f"Column '{notes_col}' not found in CSV.")

    corpus = []

    for _, row in df.iterrows():
        symptoms = preprocess_text(str(row[symptoms_col]))
        notes = preprocess_text(str(row[notes_col]))
        combined = symptoms + " " + notes
        corpus.append(combined)

    return corpus

# DEMONSTRATION BLOCK

if __name__ == "__main__":

    # 🔹 Provide dataset path externally (not hardcoded inside logic)
    FILE_PATH = r"D:\chiselon\Week 0\Week_0_Prep_Week_Ssample Data_clinic_cases.csv"

    print("Loading dataset...")
    corpus = load_corpus_from_csv(
        file_path=FILE_PATH,
        symptoms_col="symptoms",
        notes_col="doctor_notes"
    )

    print("Initializing Embedding Engine...")
    engine = EmbeddingEngine(corpus)

    print("\nGenerating embeddings for 3 sample cases:\n")

    sample_cases = [
        "Severe chest pain radiating to arm with sweating",
        "High fever with persistent cough and fatigue",
        "Severe headache with vomiting and light sensitivity"
    ]

    for i, case in enumerate(sample_cases, 1):
        embedding = engine.generate_embedding(case)

        print(f"Sample Case {i}")
        print(f"Text: {case}")
        print(f"Embedding Shape: {embedding.shape}")
        print(f"First 5 Values: {embedding[:5]}")
        print("-" * 50)

Loading dataset...
Initializing Embedding Engine...

Generating embeddings for 3 sample cases:

Sample Case 1
Text: Severe chest pain radiating to arm with sweating
Embedding Shape: (47,)
First 5 Values: [0. 0. 0. 0. 0.]
--------------------------------------------------
Sample Case 2
Text: High fever with persistent cough and fatigue
Embedding Shape: (47,)
First 5 Values: [0. 0. 0. 1. 0.]
--------------------------------------------------
Sample Case 3
Text: Severe headache with vomiting and light sensitivity
Embedding Shape: (47,)
First 5 Values: [0.         0.         0.         0.59677497 0.        ]
--------------------------------------------------


In [10]:
import os
import re
import logging
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# CONFIGURATION
CONFIG = {
    "symptoms_column": "symptoms",
    "notes_column": "doctor_notes",
    "max_features": 1000,
    "save_embeddings_path": "corpus_embeddings.npy"
}

# LOGGING SETUP
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# PREPROCESSING
def preprocess_text(text: str) -> str:
    if text is None:
        raise ValueError("Input text cannot be None.")

    if not isinstance(text, str):
        raise TypeError("Input must be a string.")

    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", "", text)

    if text == "":
        raise ValueError("Text is empty after preprocessing.")

    return text

# EMBEDDING ENGINE
class EmbeddingEngine:

    def __init__(self, corpus: list[str]):
        if not isinstance(corpus, list) or len(corpus) == 0:
            raise ValueError("Corpus must be a non-empty list.")

        logging.info("Preprocessing corpus...")
        cleaned_corpus = [preprocess_text(text) for text in corpus]

        logging.info("Initializing TF-IDF Vectorizer...")
        self.vectorizer = TfidfVectorizer(
            max_features=CONFIG["max_features"]
        )
        self.vectorizer.fit(cleaned_corpus)

        logging.info("Embedding engine initialized successfully.")

   

    def generate_embedding(self, text: str, normalize: bool = False) -> np.ndarray:
        cleaned_text = preprocess_text(text)
        vector = self.vectorizer.transform([cleaned_text]).toarray()[0]

        if normalize:
            norm = np.linalg.norm(vector)
            if norm != 0:
                vector = vector / norm

        return vector

    

    def generate_embeddings(self, texts: list[str], normalize: bool = False) -> np.ndarray:
        if not isinstance(texts, list):
            raise TypeError("Input must be a list of strings.")

        cleaned_texts = [preprocess_text(t) for t in texts]
        matrix = self.vectorizer.transform(cleaned_texts).toarray()

        if normalize:
            norms = np.linalg.norm(matrix, axis=1, keepdims=True)
            norms[norms == 0] = 1
            matrix = matrix / norms

        return matrix

   

    def get_embedding_dimension(self) -> int:
        return len(self.vectorizer.get_feature_names_out())

    

    def get_vocabulary(self) -> dict:
        return self.vectorizer.vocabulary_

    

    def save_corpus_embeddings(self, corpus: list[str]):
        matrix = self.generate_embeddings(corpus)
        np.save(CONFIG["save_embeddings_path"], matrix)
        logging.info(f"Corpus embeddings saved to {CONFIG['save_embeddings_path']}")



# DATA LOADING
def load_corpus_from_csv(file_path: str) -> list[str]:

    if not os.path.exists(file_path):
        raise FileNotFoundError("CSV file not found.")

    df = pd.read_csv(file_path)

    required_cols = [
        CONFIG["symptoms_column"],
        CONFIG["notes_column"]
    ]

    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Column '{col}' missing in dataset.")

    corpus = []

    for _, row in df.iterrows():
        symptoms = preprocess_text(str(row[CONFIG["symptoms_column"]]))
        notes = preprocess_text(str(row[CONFIG["notes_column"]]))
        combined = symptoms + " " + notes
        corpus.append(combined)

    logging.info(f"Loaded {len(corpus)} cases from dataset.")
    return corpus

# DEMONSTRATION
if __name__ == "__main__":

    FILE_PATH = r"D:\chiselon\Week 0\Week_0_Prep_Week_Ssample Data_clinic_cases.csv"

    logging.info("Loading dataset...")
    corpus = load_corpus_from_csv(FILE_PATH)

    logging.info("Initializing embedding engine...")
    engine = EmbeddingEngine(corpus)

    # Save full corpus embeddings (optional Day-2 enhancement)
    engine.save_corpus_embeddings(corpus)

    print("\nEmbedding Dimension:", engine.get_embedding_dimension())

    
    # Generated New Patient 
    
    new_patient_symptoms = "Intermittent chest tightness and shortness of breath"
    new_patient_notes = "Patient reports fatigue, mild sweating, history of diabetes."

    generated_patient_case = preprocess_text(
        new_patient_symptoms + " " + new_patient_notes
    )

    print("\n--- Processing Generated New Patient ---\n")

    new_embedding = engine.generate_embedding(
        generated_patient_case,
        normalize=True
    )

    print("New Patient Text:")
    print(generated_patient_case)
    print("\nEmbedding Shape:", new_embedding.shape)
    print("First 10 Values:", new_embedding[:10])
    print("-" * 60)

    
    # Demonstrate Batch Embedding for 3 Cases
    

    sample_cases = [
        "Severe chest pain radiating to arm with sweating",
        "High fever with persistent cough and fatigue",
        "Severe headache with vomiting and light sensitivity"
    ]

    print("\nGenerating Batch Embeddings for 3 Sample Cases...\n")

    batch_embeddings = engine.generate_embeddings(sample_cases, normalize=True)

    print("Batch Shape:", batch_embeddings.shape)
    print("First Row First 5 Values:", batch_embeddings[0][:5])
    print("-" * 60)

2026-02-19 13:59:51,271 - INFO - Loading dataset...
2026-02-19 13:59:51,398 - INFO - Loaded 10 cases from dataset.
2026-02-19 13:59:51,400 - INFO - Initializing embedding engine...
2026-02-19 13:59:51,401 - INFO - Preprocessing corpus...
2026-02-19 13:59:51,402 - INFO - Initializing TF-IDF Vectorizer...
2026-02-19 13:59:51,425 - INFO - Embedding engine initialized successfully.
2026-02-19 13:59:51,455 - INFO - Corpus embeddings saved to corpus_embeddings.npy



Embedding Dimension: 47

--- Processing Generated New Patient ---

New Patient Text:
intermittent chest tightness and shortness of breath patient reports fatigue mild sweating history of diabetes

Embedding Shape: (47,)
First 10 Values: [0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
------------------------------------------------------------

Generating Batch Embeddings for 3 Sample Cases...

Batch Shape: (3, 47)
First Row First 5 Values: [0. 0. 0. 0. 0.]
------------------------------------------------------------
